# Colab: **Master / control plane** (scheduler API)

Runs **`master.scheduler`** (FastAPI):

- **`POST /task`** → enqueue to Redis Streams (`tasks:stream`, group `workers`)
- **`GET /task/{id}`** → poll result / meta
- **`GET /health`** → Redis ping

**Redis** must exist somewhere this runtime can reach:

- **Option A**: Start **Redis locally** on this Colab VM (cells below)—good for a quick single-session demo (**Colab worker notebooks** still need network access to this Redis unless you expose it—usually you run workers on VPS or use tunnels).
- **Option B**: **`REDIS_URL`** points to **Redis on your VPS** (typical prod): scheduler can run Colab **or** VPS; workers use the **same** `REDIS_URL`.

If clients or workers are **off** this machine, expose **`8000`** (ngrok / Cloudflare Tunnel / Colab “open port” features) and open Redis only behind a firewall or VPN—not public without auth.

---

**Match with workers**: Workers do **not** call this HTTP API unless you add that; they only need **Redis**. Set **`BASE_URL`** in **`02_Connectivity_Redis_API.ipynb`** / load generator to this API’s URL when testing.

## 1) Project root

`git clone` or upload/unzip under `/content/`.

In [ ]:
import os

# !git clone https://github.com/YOUR_ORG/YOUR_REPO.git /content/dist

PROJECT_ROOT = "/content/dist"
os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())
assert os.path.isfile("main.py") and os.path.isdir("master"), "Fix PROJECT_ROOT"

## 2) Install **master** Python deps

**`requirements-master-api.txt`** (no Ray / FAISS / transformers).

In [ ]:
%pip install -q -r requirements-master-api.txt

## 3) Configure **Redis**

Set **`USE_LOCAL_REDIS = True`** to install and launch **`redis-server`** on Colab (`redis://127.0.0.1:6379/0`).

Set **`USE_LOCAL_REDIS = False`** to type a remote **`REDIS_URL`** (e.g. `redis://:password@vps-host:6379/0`).

In [ ]:
import os
import subprocess
from getpass import getpass

os.chdir(PROJECT_ROOT)

USE_LOCAL_REDIS = True  # <-- False if using VPS / managed Redis

if USE_LOCAL_REDIS:
    subprocess.run(
        ["bash", "-lc", "sudo apt-get update -qq && sudo apt-get install -y -qq redis-server"],
        check=True,
    )
    subprocess.run(["bash", "-lc", "redis-server --daemonize yes"], check=True)
    subprocess.run(["bash", "-lc", "sleep 1 && redis-cli ping"], check=True)
    os.environ["REDIS_URL"] = "redis://127.0.0.1:6379/0"
    print("REDIS_URL:", os.environ["REDIS_URL"])
else:
    os.environ["REDIS_URL"] = os.environ.get("REDIS_URL") or getpass("REDIS_URL: ")
    print("Using remote REDIS_URL (host hidden)")

## 4) Quick Redis + stream check

Consumer group **`workers`** is created when the FastAPI app starts; this cell only pings Redis.

In [ ]:
import os
import redis

r = redis.from_url(os.environ["REDIS_URL"], decode_responses=True)
print("PING:", r.ping())

## 5) Run the **scheduler API** (blocks)

Listens on **`0.0.0.0:8000`**. Stop with Colab’s stop button.

Equivalent: `PYTHONPATH=. uvicorn master.scheduler:app --host 0.0.0.0 --port 8000`.

**Expose to the internet**: add your own tunnel in a separate cell after starting the server in the background—or run this notebook on a VPS behind **`nginx/nginx.conf`** in the repo.

In [ ]:
import os
import sys

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import uvicorn

uvicorn.run(
    "master.scheduler:app",
    host="0.0.0.0",
    port=8000,
    reload=False,
)

## 6) (Optional) Smoke **POST /task** from this notebook

**Run after** starting the API in the **background** (not included here). Example with `requests`:

```python
import requests
r = requests.post("http://127.0.0.1:8000/task", json={"query": "Hello from Colab master"})
r.raise_for_status()
print(r.json())
```